# 02 — Spark Data Processing (7 JP)

**Pelatihan Pengolahan Data dengan Spark — TA 2026**

Data latihan bersifat sintetis, bukan data wajib pajak sebenarnya.


**Split:** Day 1: 4 JP; Day 2: 3 JP. Theory 2 JP; practice 5 JP.

Objectives: SparkSession, schemas, reading CSV, filter, sort, select, columns, null handling.


In [ ]:
import os
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder
    .appName("SparkTraining-Notebook")
    .master(os.getenv("SPARK_MASTER", "spark://spark-master:7077"))
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4050")
    .config("spark.blockManager.port", "4051")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
print("Spark:", spark.version, "master:", spark.sparkContext.master)


In [ ]:
taxpayers = (spark.read.option("header", True).option("inferSchema", True)
    .csv("/home/spark/datasets/taxpayers.csv"))
taxpayers.printSchema(); taxpayers.show(10, truncate=False)
print("Rows:", taxpayers.count())


In [ ]:
taxpayers.select("taxpayer_id", "region", "sector").filter(
    (F.col("region") == "Jakarta") & (F.col("active") == True)
).orderBy("taxpayer_id").show(15, False)


In [ ]:
normalized = (taxpayers.withColumn("region_upper", F.upper(F.trim(F.col("region"))))
  .withColumn("registration_age", F.lit(2026) - F.col("registration_year"))
  .fillna({"sector": "Unknown"}))
normalized.groupBy("region").agg(F.count("*").alias("n"),F.avg("registration_age").alias("avg_age")).orderBy(F.desc("n")).show()


### Exercises
1. Show active taxpayers registered after 2021, sorted by year descending.
2. Create an `age_group` column with `when`.
3. Count taxpayers by region **and** sector.
4. Compare inferred schema to an explicit `StructType`.
5. Why is `collect()` dangerous for very large DataFrames?

**Deliverable:** notebook with transformations + row count checks.


In [ ]:
# TODO for learners: age_group = normalized.withColumn(...)
print("Expected taxpayer rows:", taxpayers.count(), "(300)")
